# 01 Experiment 1: Low-Target-Data Transfer

Runs the low-target-data experiment. Each setting is independent and resumable.

## Colab setup notes

1. In Colab, choose **Runtime -> Change runtime type -> GPU or Premium GPU**.
2. Optionally mount Google Drive for persistent `results_T500/` outputs.
3. The code below checks `torch.cuda.is_available()` and prints the GPU name when available.
4. Training automatically uses CUDA if available, otherwise CPU.
5. Results, configs, logs, and checkpoints are saved frequently so interrupted runs can resume.


In [ ]:
# Optional in Colab: mount Drive for persistent outputs. Uncomment if desired.
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import sys

# If running from a cloned repo in Colab, set PROJECT_ROOT to that clone path.
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%pip install -q -r requirements.txt


In [ ]:
import torch
print('torch.cuda.is_available():', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)


In [ ]:
from config import default_experiment1_configs
from train import run_single_setting

configs = default_experiment1_configs(seeds=[0, 1, 2])
for cfg in configs:
    cfg.training.device = DEVICE
    cfg.training.training_steps = 20_000
    cfg.training.resume_from_checkpoint = False
    cfg.results_dir = Path('results_T500')
    # Set True only if you want PCA sample files; they can be large for full runs.
    cfg.evaluation.save_samples = False

print(f'Prepared {len(configs)} settings.')


In [ ]:
# Resume-friendly loop: partial metrics are appended after each model/setting.
for i, cfg in enumerate(configs, start=1):
    print(f'Running setting {i}/{len(configs)}:', cfg.data.covariance_scenario, cfg.data.rho, cfg.data.mismatch_level, cfg.n_target_train, 'seed', cfg.seed)
    run_single_setting(cfg, force=False)
